XÂY DỰNG MÔ HÌNH SVM

In [20]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

processed_dir = Path(
    r"c:\Users\hauth\OneDrive\Tài liệu\GitHub\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\BÀI TẬP NHÓM HẰNG TUẦN\LAB5 BÀI TOÁN PHÂN LỚP (NHÓM)\THUẬT TOÁN PHÂN LỚP\2.2.3 diabetes\processed"
)

if not processed_dir.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục: {processed_dir}")

csv_files = list(processed_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Không tìm thấy file CSV trong: {processed_dir}")

target = "Outcome"
train_files = [p for p in csv_files if "train" in p.stem.lower()]
test_files = [p for p in csv_files if "test" in p.stem.lower()]

if train_files and test_files:
    train_df = pd.read_csv(train_files[0])
    test_df = pd.read_csv(test_files[0])

    if target not in train_df.columns or target not in test_df.columns:
        x_train_file = next((p for p in csv_files if p.stem.lower() == "x_train"), None)
        x_test_file = next((p for p in csv_files if p.stem.lower() == "x_test"), None)
        y_train_file = next((p for p in csv_files if p.stem.lower() == "y_train"), None)
        y_test_file = next((p for p in csv_files if p.stem.lower() == "y_test"), None)

        if not all((x_train_file, x_test_file, y_train_file, y_test_file)):
            raise ValueError("Không tìm thấy đủ các file X_train, X_test, y_train và y_test.")

        train_df = pd.read_csv(x_train_file)
        test_df = pd.read_csv(x_test_file)
        y_train_df = pd.read_csv(y_train_file)
        y_test_df = pd.read_csv(y_test_file)

        train_labels = (
            y_train_df[target] if target in y_train_df.columns else y_train_df.iloc[:, 0]
        )
        test_labels = (
            y_test_df[target] if target in y_test_df.columns else y_test_df.iloc[:, 0]
        )

        if len(train_df) != len(train_labels) or len(test_df) != len(test_labels):
            raise ValueError("Số dòng trong file X và y không khớp.")

        train_df[target] = train_labels.to_numpy()
        test_df[target] = test_labels.to_numpy()

    X_train = train_df.drop(columns=target)
    y_train = train_df[target]
    X_test = test_df.drop(columns=target).reindex(columns=X_train.columns)
    y_test = test_df[target]
else:
    if len(csv_files) != 1:
        raise ValueError(
            "Không xác định được file train/test. Hãy đặt tên file có chữ "
            "'train' và 'test', hoặc chỉ để một file CSV trong processed."
        )

    df = pd.read_csv(csv_files[0])
    if target not in df.columns:
        raise ValueError(f"Không tìm thấy cột nhãn '{target}' trong {csv_files[0].name}")

    X = df.drop(columns=target)
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

svm_model = make_pipeline(
    StandardScaler(),
    LinearSVC(C=1.0, random_state=42, max_iter=10000)
)
svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.8812272490899636
              precision    recall  f1-score   support

           0       0.99      0.88      0.93     17534
           1       0.42      0.88      0.57      1696

    accuracy                           0.88     19230
   macro avg       0.70      0.88      0.75     19230
weighted avg       0.94      0.88      0.90     19230



- Tinh chỉnh tham số

In [30]:
linear_svm_param_grid = {
    "linearsvc__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "linearsvc__class_weight": ["balanced"],
}

In [31]:
linear_svm_grid_search.fit(X_train, y_train)

svm_model = linear_svm_grid_search.best_estimator_

print("Tham số tốt nhất:", linear_svm_grid_search.best_params_)
print("F1-score trung bình trên tập validation:", linear_svm_grid_search.best_score_)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Tham số tốt nhất: {'linearsvc__C': 0.001, 'linearsvc__class_weight': 'balanced'}
F1-score trung bình trên tập validation: 0.8843855576391565


- Đánh giá

In [33]:
from sklearn.metrics import classification_report, accuracy_score

models = {
    "Trước tinh chỉnh": baseline_pred,
    "Sau tinh chỉnh": tuned_pred,
}

comparison_df = pd.DataFrame([
    {
        "Mô hình": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (lớp 1)": classification_report(
            y_test, pred, output_dict=True, zero_division=0
        )["1"]["precision"],
        "Recall (lớp 1)": classification_report(
            y_test, pred, output_dict=True, zero_division=0
        )["1"]["recall"],
        "F1-score (lớp 1)": classification_report(
            y_test, pred, output_dict=True, zero_division=0
        )["1"]["f1-score"],
    }
    for name, pred in models.items()
])

display(comparison_df.round(4))

,Mô hình,Accuracy,Precision (lớp 1),Recall (lớp 1),F1-score (lớp 1)
0,Trước tinh chỉnh,0.8812,0.4176,0.8785,0.5661
1,Sau tinh chỉnh,0.8812,0.4176,0.8785,0.5661
